# Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# sklearn config
from sklearn import set_config
set_config(
    transform_output='pandas',  # `transform` and `fit_transform` returns pd.DataFrame.
    display='text',             # `pipeline` properly prints into the jupyter output cell.
)

# Data

In [2]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 1. Naive Bayes

- A suitable baseline for many tasks, e.g. text classification.
- Extremely fast.
- Usable with very small datasets.
- Assume independency between features.

## 1.1. GaussianNB

- Continuous features.
- Suppose normal distribution.

In [3]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

model = GaussianNB()
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print("Prediction:", model.predict(X_test[:1]))
print("Probability:", model.predict_proba(X_test[:1]))

Accuracy: 0.9666666666666667
Prediction: [0]
Probability: [[1.0000000e+00 2.1042361e-20 1.2404669e-25]]


## 1.2. MultinomialNB

- Count frequencies of categorical features.

In [4]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

texts = [
    "Sharing the meeting schedule",
    "Please review the project documents",
    "Submitting the work report",
    "Please respond to the customer inquiry",
    "Please choose the lunch menu",
    "Free prize winner click now",
    "Loan approval promotion click link",
]

# 0: normal, 1: spam
y = [0, 0, 0, 0, 0, 1, 1]

model = make_pipeline(
    CountVectorizer(),             # number of appearance.
    MultinomialNB(alpha=1.0),      # Laplace smoothing.
)

model.fit(texts, y)

test_text = ["Free promotion prize link click"]

print("Prediction:", model.predict(test_text))
print("Probability:", model.predict_proba(test_text))

Prediction: [1]
Probability: [[0.01035939 0.98964061]]


## 1.3. BernoulliNB

- Only consider existence.

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import BernoulliNB
from sklearn.pipeline import make_pipeline

model = make_pipeline(
    CountVectorizer(binary=True),  # 1: existence, 0: not existence.
    BernoulliNB(alpha=1.0),
)

model.fit(texts, y)

test_text = ["Free free free promotion click"]

print("Prediction:", model.predict(test_text))
print("Probability:", model.predict_proba(test_text))

Prediction: [1]
Probability: [[0.04688799 0.95311201]]


## 1.4. ComplementNB

- Same as `MultinomialNB`, but consider **other** classes.

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import make_pipeline

model = make_pipeline(
    CountVectorizer(),
    ComplementNB(alpha=1.0),
)

model.fit(texts, y)

test_text = ["Free loan offer click now"]

print("Prediction:", model.predict(test_text))
print("Probability:", model.predict_proba(test_text))

Prediction: [1]
Probability: [[0.01141118 0.98858882]]


# 2. Gaussian Process

- Consider when a dataset is **small**, and a **certainty** of prediction matters.

In [7]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF, Matern, WhiteKernel, ConstantKernel
)

# Define the signal and observation noise.
kernel = (
    ConstantKernel(1.0)
    * Matern(length_scale=1.0, nu=2.5)
    + WhiteKernel(noise_level=0.1)
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,  # Add numerical stability or extra observation noise.
    normalize_y=True,
    n_restarts_optimizer=5,
    random_state=42,
)

gp.fit(X_train, y_train)

# Predict the mean and standard deviation.
y_mean, y_std = gp.predict(X_test, return_std=True)

# Predict the mean and full covariance matrix.
y_mean, y_cov = gp.predict(X_test, return_cov=True)

# Draw function samples from the posterior.
y_samples = gp.sample_y(X_test, n_samples=10)

# Inspect the main fitted attributes.
gp.kernel_                         # Optimized kernel.
gp.kernel_.theta                   # Optimized kernel parameters.
gp.log_marginal_likelihood_value_  # Optimized log-marginal likelihood.
gp.X_train_
gp.y_train_

array([-1.22474487,  1.22474487,  0.        , -1.22474487,  0.        ,
        1.22474487,  0.        ,  1.22474487,  1.22474487,  1.22474487,
        1.22474487,  0.        ,  0.        ,  0.        ,  0.        ,
       -1.22474487, -1.22474487,  1.22474487,  1.22474487, -1.22474487,
        0.        , -1.22474487,  1.22474487, -1.22474487,  0.        ,
        1.22474487,  1.22474487, -1.22474487,  1.22474487, -1.22474487,
       -1.22474487,  0.        ,  0.        , -1.22474487,  1.22474487,
        1.22474487,  0.        ,  0.        ,  1.22474487,  0.        ,
       -1.22474487,  0.        , -1.22474487,  1.22474487, -1.22474487,
       -1.22474487,  1.22474487, -1.22474487, -1.22474487, -1.22474487,
       -1.22474487,  0.        ,  1.22474487,  0.        , -1.22474487,
        1.22474487,  0.        ,  1.22474487, -1.22474487,  1.22474487,
       -1.22474487,  0.        ,  1.22474487, -1.22474487,  0.        ,
        0.        ,  1.22474487,  0.        ,  0.        ,  1.22